# Standard Types & Custom Enums
*Fabian Nütz*

Standard types and enums define the **vocabulary** of your network. Customizing them lets you model exactly the technologies, protocols, and device/app classes you need. This notebook covers:

- Creating and modifying **link types** (bandwidth, latency, reliability params)
- Creating **protocol types** (supported info items)
- Extending **enums** at runtime with `addCustomEnum()`
- Setting per-element reliability overrides
- How std_type values propagate into calculations

## Setup

In [2]:
import pandaict as pi
from pandaict.utils.enums import NodeType, DeviceType, AppType, InfoItem, addCustomEnum

## Link types

Link types define the physical characteristics of a connection technology. Every link in your network references one of these types.

**Required fields for links**:
- `bandwidth_Mbps` — tuple `(downstream, upstream)` for bidirectional links or single float for unidirectional
- `latency_ms` — baseline latency in milliseconds. Tuple `(downstream, upstream)`or single float for symmetric
- `wired` — `True` for wired, `False` for wireless

**Recommended fields** (used by reliability calculations):
- `protocols` — list of protocol names this link supports
- `mtbf_h` — Mean Time Between Failure (hours)
- `mttr_h` — Mean Time To Repair (hours)
- `ber` — Bit Error Rate per km

In [3]:
net = pi.create_empty_network(name='Custom Types Demo')

# View built-in link types
print('Built-in link types:', list(net.std_types['link'].keys()))

Built-in link types: ['Binary', 'RS485', 'LAN1000', 'LTE50', 'UMTS', 'ADSL8', 'Fiber']


In [4]:
# Create a custom fiber link type
pi.create_std_type(net, {
    'bandwidth_Mbps': (10000, 10000),   # 10 Gbps symmetric, bidirectional
    'latency_ms': 0.5,
    'wired': True,
    'protocols': ['TCP', 'UDP'],
    'mtbf_h': 43800,                     # 5 years
    'mttr_h': 4,
    'ber': 1e-6
}, name='Fiber10G', element='link')

# Create a LoRaWAN wireless link type (asymmetric bandwidth)
pi.create_std_type(net, {
    'bandwidth_Mbps': (0.05, 0.01),     # 50 kbps down, 10 kbps up
    'latency_ms': 200,
    'wired': False,
    'protocols': ['TCP'],
    'mtbf_h': 8760,                      # 1 year
    'mttr_h': 8
}, name='LoRaWAN', element='link')

print('Updated link types:', list(net.std_types['link'].keys()))

Updated link types: ['Binary', 'RS485', 'LAN1000', 'LTE50', 'UMTS', 'ADSL8', 'Fiber', 'Fiber10G', 'LoRaWAN']


In [5]:
# Inspect our new types
print('Fiber10G:')
for k, v in net.std_types['link']['Fiber10G'].items():
    print(f'  {k}: {v}')

print('\nLoRaWAN:')
for k, v in net.std_types['link']['LoRaWAN'].items():
    print(f'  {k}: {v}')

Fiber10G:
  bandwidth_Mbps: (10000, 10000)
  latency_ms: 0.5
  wired: True
  protocols: ['TCP', 'UDP']
  mtbf_h: 43800
  mttr_h: 4
  ber: 1e-06
  custom: True

LoRaWAN:
  bandwidth_Mbps: (0.05, 0.01)
  latency_ms: 200
  wired: False
  protocols: ['TCP']
  mtbf_h: 8760
  mttr_h: 8
  custom: True


In [6]:
# Use the custom types in a network
n1 = pi.create_ict_node(net, name='Node1', node_type=NodeType.SWITCH, link_types=['Fiber10G'])
n2 = pi.create_ict_node(net, name='Node2', node_type=NodeType.ROUTER, link_types=['Fiber10G', 'LoRaWAN'])

d1 = pi.create_ict_device(net, name='Server1', device_type=DeviceType.SERVER)
d2 = pi.create_ict_device(net, name='RTU1',    device_type=DeviceType.RTU)

pi.create_ict_link(net, name='Fib-Link',  from_elt=n1, to_elt=[n2], link_type='Fiber10G', length_km=5.0)
pi.create_ict_link(net, name='LTU-Link',  from_elt=n2, to_elt=[d2], link_type='LoRaWAN')
pi.create_ict_link(net, name='Serv-Link', from_elt=d1, to_elt=[n1], link_type='Fiber10G')

display(net.link[['name', 'type', 'length_km']])
print('\nNetwork built with custom link types.')

,name,type,length_km
0,Fib-Link,Fiber10G,5.0
1,LTU-Link,LoRaWAN,0.0
2,Serv-Link,Fiber10G,0.0



Network built with custom link types.


Note that if a standard type is created with the same name as an already existing one, the existing will be overwritten. This is also the case for the base standard types. Standard types created at runtime will also be tagged as custom, and as such be included in the JSON or Excel files by default when the network is saved. 

### Batch creation with `create_std_types()`

In [7]:
pi.create_std_types(net, {
    'FastEthernet': {'bandwidth_Mbps': (100, 100), 'latency_ms': 0.1, 'wired': True,
                    'protocols': ['TCP'], 'mtbf_h': 50000, 'mttr_h': 2},
    '5G-NR':        {'bandwidth_Mbps': (1000, 150), 'latency_ms': 10, 'wired': False,
                    'protocols': ['TCP', 'UDP'], 'mtbf_h': 20000, 'mttr_h': 1},
}, element='link')

print('Batch types created:', [t for t in net.std_types['link'].keys() if t in ('FastEthernet', '5G-NR')])

Batch types created: ['FastEthernet', '5G-NR']


Again, existing types are overwritten with new ones if the name is the same. 

## Protocol types

Protocol types define what communication protocols are available. 

**Required fields for protocols**:

- `base_package_size_byte` - minimal package size in byte.
- `info_items` - a list of `InfoItem` enums that can be carried by this protocol.

**Optional fields for protocols**:

- `payload_size_byte` - list of payload sizes in byte. Each entry corresponds to the entry in `info_items` with same index, thus both lists must have the same length. The total package size is the sum of the payload and the base package size.


In [8]:
print('Built-in protocol types:', list(net.std_types['protocol'].keys()))

Built-in protocol types: ['Binary', 'TCP', 'UDP', 'HTTP', 'HTTPS', 'FTP', 'MODBUS-TCP', 'MODBUS-SMA']


In [10]:
pi.create_std_type(net, {
    'base_package_size_byte': 64,
    'info_items': [InfoItem.VALUE, InfoItem.STATUS, InfoItem.ACK]
}, name='IEC61850-MMS', element='protocol')

print('IEC61850-MMS:', net.std_types['protocol']['IEC61850-MMS'])

IEC61850-MMS: {'base_package_size_byte': 64, 'info_items': [<InfoItem.VALUE: 2>, <InfoItem.STATUS: 5>, <InfoItem.ACK: 1>], 'custom': True}


Again, existing types are overwritten with new ones if the name is the same. Bulk creation of protocol types is possible as well, using the function `create_std_types` in the same way as for link type bulk creation.

## Custom enums with `addCustomEnum()`

pandaict uses the `aenum` library, which allows extending enum classes **at runtime**. The function `addCustomEnum()` can register a new value in one of the four built-in enums: `NodeType`, `DeviceType`, `AppType`, `InfoItem`.

In [9]:
# Add a new node type
addCustomEnum(net, NodeType, 'REPEATER')

# Add a new device type
addCustomEnum(net, DeviceType, 'PROTECTION_RELAY')

# Add a new app type
addCustomEnum(net, AppType, 'SCADA_MASTER')

# Add a new info item
addCustomEnum(net, InfoItem, 'POWER_SETPOINT')

print('Custom enums added.')
print('NodeType.REPEATER:', NodeType.REPEATER)
print('InfoItem.POWER_SETPOINT:', InfoItem.POWER_SETPOINT)

Custom enums added.
NodeType.REPEATER: NodeType.REPEATER
InfoItem.POWER_SETPOINT: InfoItem.POWER_SETPOINT


Once defined, custom enums may be used just like pre-defined ones: 

In [10]:
# Use the custom enums in element creation
n3 = pi.create_ict_node(net, name='Repeater1', node_type=NodeType.REPEATER, link_types=['Fiber10G'])
rd = pi.create_ict_device(net, name='Relay1', device_type=DeviceType.PROTECTION_RELAY)

display(net.node[net.node['type'] == NodeType.REPEATER][['name', 'type']])
display(net.device[net.device['type'] == DeviceType.PROTECTION_RELAY][['name', 'type']])

,name,type
2,Repeater1,NodeType.REPEATER


,name,type
2,Relay1,DeviceType.PROTECTION_RELAY


**Note**: Custom enum values are auto-assigned starting from 1000 (to avoid collisions with built-in values). You can also pass an explicit `value=` to `addCustomEnum()`. If a custom enum is added with the same name as an existing one, the existing will be retained and not be assigned a new value. All custom defined enums are kept in an internal list, and saved as well as loaded to and from JSON or Excel files. 

## How std_type values propagate

When functions like `calc_mtbf_route()` or `calc_ber_route()` run, they look up values in this priority order:

1. **Per-element override** (set directly on the node / device. For links, the parameters are given through the linktype.)
2. **Standard type default** (`net.std_types['link'][link_type]`)
3. **Built-in fallback** (e.g., `0.0` for BER)

This means you can set sensible defaults in std_types and override only where needed.

## Summary

| Task | Function |
|------|----------|
| Add/modify a link type | `create_std_type(net, {...}, name='...', element='link')` |
| Add/modify a protocol type | `create_std_type(net, {...}, name='...', element='protocol')` |
| Batch add types | `create_std_types(net, {...}, element='...')` |
| Extend an enum | `addCustomEnum(net, EnumClass, 'NEW_VALUE')` |

Next: [07 – Advanced Topics](07_advanced_topics.ipynb)